# Human Niche-Identification Benchmark: TERRA vs. Baselines

- Kernel: the `terra` environment (`/nfs/team361/sb75/.venvs/terra/bin/python`). The notebook only reads results; nothing is recomputed.
- Inputs: per-method results in `artifacts/benchmarking/human/<method>/` (written by the LSF jobs, see `README.md`), collected by `collect_results.py` into `artifacts/benchmarking/human/summary/`.
- Protocol: identical cells for every method; per-section spatial graphs; Leiden on a 15-NN graph of each embedding, one upward scan on a 0.01 resolution grid with every run scored. **Headline** = lowest grid resolution giving the ground-truth number of niches (niches with > 100 cells); **matched** = all runs giving it; **sweep** = ±4 grid steps around the median matching resolution. NMI/ARI vs. ground truth.
- Banksy = BANKSY + Harmony (forced post-hoc integration). Novae follows the paper's mouse benchmark: Leiden on the Novae embedding after Novae's batch-effect correction based on its native domains. ‡ Nicheformer on the ISS reproductive tract cannot reach 19 clusters (its 15-NN graph splits into 211 disconnected components); its best reachable run is shown.

In [ ]:
import sys
sys.path.append('.')

import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc

import collect_results
import plotting as P
from bench_utils import DATASET_LABELS, METHODS

SUMMARY = collect_results.OUT
RESULTS = collect_results.RESULTS
FIG_DIR = SUMMARY / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'svg.fonttype': 'none', 'figure.dpi': 110})
UNREACHABLE = {('nicheformer', 'chr78-11b_1p'): '‡'}

## 1. Collect Results

In [ ]:
# Re-collect so the notebook always reflects the latest finished jobs
collect_results.main()
headline = pd.read_csv(SUMMARY / 'headline.csv')
sweep = pd.read_csv(SUMMARY / 'sweep.csv')
matched = pd.read_csv(SUMMARY / 'matched.csv')
batch = pd.read_csv(SUMMARY / 'batch.csv')
labels = [spec['label'] for spec in METHODS.values()]

In [ ]:
# Headline run per method and dataset; matched mean = mean over all runs giving
# exactly the ground-truth number of niches
display(headline.pivot_table(index='method_name', columns='dataset_label',
                             values=['NMI', 'ARI']).reindex(labels).round(3))
display(headline[['method_name', 'dataset_label', 'n_clusters', 'n_target', 'resolution',
                  'NMI', 'ARI', 'n_matched_resolutions', 'NMI_matched_mean',
                  'ARI_matched_mean', 'note']].round(3))

## 2. Main Figure: NMI and ARI (Paper Convention)

Same convention as the paper's mouse figure (`mouse_benchmarking/benchmarking_comparison.ipynb`): one panel per dataset; bar = mean over the resolution sweep, error bar = 1 SD, dots = individual sweep runs, dashed line = TERRA mean; method colours, labels and order as in the paper.

In [ ]:
with plt.rc_context(P.PAPER_RC):
    for metric in ['NMI', 'ARI']:
        fig = P.plot_paper_metric_comparison(sweep, metric, DATASET_LABELS,
                                             annotations=UNREACHABLE)
        for ext in ['svg', 'pdf', 'png']:
            fig.savefig(FIG_DIR / f'niche_{metric}_comparison_human.{ext}')
        plt.show()

### 2b. Matched Number of Clusters Only

Same figure with only the runs that give exactly the ground-truth number of niches (all of them from the scan, not only those in the sweep window): bar = mean over these runs, error bar = 1 SD, dots = these runs. Methods hitting the target at a single grid resolution show one dot and no error bar.

In [ ]:
# Number of matched runs per method and dataset
display(matched.groupby(['method_name', 'dataset_label']).size().unstack().reindex(labels))
with plt.rc_context(P.PAPER_RC):
    for metric in ['NMI', 'ARI']:
        fig = P.plot_paper_metric_comparison(matched, metric, DATASET_LABELS,
                                             annotations=UNREACHABLE)
        for ext in ['svg', 'pdf', 'png']:
            fig.savefig(FIG_DIR / f'niche_{metric}_comparison_human_matched.{ext}')
        plt.show()

## 3. Overview Heatmap (Headline Run)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
for ax, metric in zip(axes, ['NMI', 'ARI']):
    im = P.plot_heatmap(headline, metric, DATASET_LABELS, ax)
axes[1].set_yticklabels([])
fig.colorbar(im, ax=axes, shrink=0.6, label='Score')
fig.savefig(FIG_DIR / 'niche_identification_heatmap.svg', bbox_inches='tight')
plt.show()

## 4. Robustness: Scores Around the Target Number of Clusters

Each line is one method's sweep (±4 grid steps around its median matching resolution); dashed line = ground-truth number of niches. TERRA is drawn thick.

In [ ]:
for metric in ['NMI', 'ARI']:
    fig, axes = plt.subplots(1, len(DATASET_LABELS), figsize=(16, 3.6))
    P.plot_sweep(sweep, metric, DATASET_LABELS, axes)
    fig.tight_layout()
    fig.savefig(FIG_DIR / f'niche_identification_sweep_{metric.lower()}.svg',
                bbox_inches='tight')
    plt.show()

## 5. Batch Integration (Multi-Section Datasets)

iLISI (higher = sections better mixed), batch ASW (closer to 0 = better mixed), MMD averaged over section pairs (lower = better mixed). Mixing sections is only desirable to the extent that the niches are shared across sections.

In [ ]:
display(batch.pivot_table(index='method_name', columns=['dataset_label', 'metric'],
                          values='score').reindex(labels).round(3))

## 6. Runtime and Coverage

In [ ]:
display(headline.pivot_table(index='method_name', columns='dataset_label',
                             values='embedding_minutes').reindex(labels).round(1))
display(headline.pivot_table(index='method_name', columns='dataset_label',
                             values='coverage').reindex(labels).round(3))

## 7. Spatial Maps: Ground Truth vs. Methods (Headline Clustering)

Predicted clusters take the colour of the ground-truth niche they overlap most (Hungarian matching over the whole dataset); unmatched clusters are grey. Large sections are subsampled for plotting only.

In [ ]:
def load_dataset_results(dataset):
    adatas = {}
    for method, spec in METHODS.items():
        path = RESULTS / method / f'{dataset}_{method}_embedded_clustered.h5ad'
        if path.exists():
            adatas[spec['label']] = sc.read_h5ad(path)
    return adatas


for dataset, label in DATASET_LABELS.items():
    adatas = load_dataset_results(dataset)
    ref = next(iter(adatas.values()))
    gt_cats = ref.obs['niche_gt'].astype(str).value_counts().index.tolist()
    gt_colors = P.niche_colors(gt_cats)
    sections = ref.obs['section'].cat.categories.tolist()
    n_cols = len(adatas) + 1
    fig, axes = plt.subplots(len(sections), n_cols,
                             figsize=(2.2 * n_cols, 2.3 * len(sections)), squeeze=False)
    for row, section in zip(axes, sections):
        P.plot_spatial_panels(adatas, section, row, gt_colors, max_cells=60000)
    handles = [plt.Line2D([], [], marker='o', linestyle='', color=c, label=n)
               for n, c in gt_colors.items()]
    fig.legend(handles=handles, loc='center left', bbox_to_anchor=(1.0, 0.5),
               fontsize=7, frameon=False, title=f'{label}: ground-truth niches',
               title_fontsize=8)
    fig.tight_layout()
    fig.savefig(FIG_DIR / f'spatial_maps_{dataset}.png', dpi=200, bbox_inches='tight')
    plt.show()
    del adatas